# Iteración V2: SARIMAX Avanzado (Log-Trans y Walk-Forward)

Para mejorar el modelo ARIMAX estadístico anterior, vamos a centrarnos en estabilizar su principal debilidad: **La Varianza y los Outliers**.

1. **Transformación Logarítmica (`np.log1p`):** Aplicar esto a Gastos e Ingresos suaviza matemáticamente los picos extremos del verano, haciendo que la serie temporal sea más fácil de predecir estadísticamente.
2. **Walk-Forward Validation:** En vez de hacer un simple corte al 80%, simularemos cómo predeciríamos mes a mes en la vida real. El modelo entrena hasta T, predice T+1. Luego añade el dato real T+1, se reentrena (o actualiza) y predice T+2.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pmdarima import auto_arima
import warnings

warnings.filterwarnings('ignore')

### 1. Carga, Limpieza y Transformación Logarítmica

In [ ]:
# Cargar datos
file_path = '../data/raw/db_orig.csv'
df = pd.read_csv(file_path)

df['Amount'] = df['Amount'].str.replace('€', '', regex=False).str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
df['Amount'] = pd.to_numeric(df['Amount'])
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

# Agrupar Mensual (Month End)
df['Month_End'] = df['Date'].dt.to_period('M').dt.to_timestamp('M')
monthly = df.groupby(['Month_End', 'Type'])['Amount'].sum().unstack(fill_value=0).reset_index()

if 'Expenses' not in monthly.columns: monthly['Expenses'] = 0
if 'Income' not in monthly.columns: monthly['Income'] = 0

ts_df = monthly.set_index('Month_End').sort_index()
ts_df.index.freq = 'ME'

### Novedad: Suavizado Logarítmico Log1p
# Usamos log1p (log(1 + x)) porque maneja los ceros perfectamente en caso de que no haya ingresos un mes
ts_df['Expenses_Log'] = np.log1p(ts_df['Expenses'])
ts_df['Income_Log'] = np.log1p(ts_df['Income'])

# Feature Enginering Extra para Exógena: Rolling Log Income (Inercia de Ingresos Suavizada)
ts_df['Income_Rolling_3_Log'] = ts_df['Income_Log'].shift(1).rolling(3).mean().fillna(method='bfill')

display(ts_df[['Expenses', 'Expenses_Log', 'Income', 'Income_Log']].head())

### 2. Buscar Parámetros Óptimos sobre la serie Logarítmica

In [ ]:
y_log = ts_df['Expenses_Log']
exog_log = ts_df[['Income_Log', 'Income_Rolling_3_Log']]

print("Buscando los mejores parámetros para la serie estabilizada...")
## Mantenemos estacionalidad mensual pequeña (m=6) o anual (m=12)
auto_model = auto_arima(y_log, X=exog_log, seasonal=True, m=6,
                        start_p=0, start_q=0, max_p=3, max_q=3,
                        d=None, D=None, trace=False,
                        error_action='ignore', suppress_warnings=True, stepwise=True)

print(auto_model.summary().tables[1])
optimal_order = auto_model.order
optimal_seasonal = auto_model.seasonal_order

### 3. Evaluación Continua (Walk-Forward Validation)
Simularemos el último año mes a mes. Empezamos con el historial hasta T, predecimos T+1 y guardamos. Luego actualizamos el historial y repetimos.

In [ ]:
split_idx = int(len(ts_df) * 0.8)

train_y_log = y_log.iloc[:split_idx]
train_exog_log = exog_log.iloc[:split_idx]

test_y_log = y_log.iloc[split_idx:]
test_exog_log = exog_log.iloc[split_idx:]
test_y_real = ts_df['Expenses'].iloc[split_idx:] # La verdad para calcular el error final (€)

history_y = list(train_y_log)
history_exog = [row for index, row in train_exog_log.iterrows()] # Lista de Series (filas)
predictions_log = []

print("Iniciando Walk-Forward Validation...")

for t in range(len(test_y_log)):
    # Definir el modelo con la historia HASTA este punto
    model = SARIMAX(endog=history_y, exog=pd.DataFrame(history_exog),
                    order=optimal_order, seasonal_order=optimal_seasonal,
                    enforce_stationarity=False, enforce_invertibility=False)
    
    model_fit = model.fit(disp=False)
    
    # Predecir 1 paso adelante pasándole las exógenas del siguiente paso
    next_exog = test_exog_log.iloc[[t]]
    yhat_log = model_fit.forecast(steps=1, exog=next_exog).iloc[0]
    predictions_log.append(yhat_log)
    
    # Agregar la observación real del paso t a la historia para la siguiente iteración
    history_y.append(test_y_log.iloc[t])
    history_exog.append(test_exog_log.iloc[t])

# Revertir la transformación logarítmica para ver los errores en Euros reales (expm1 revive log1p)
predictions_real = np.expm1(predictions_log)

mae = mean_absolute_error(test_y_real, predictions_real)
rmse = np.sqrt(mean_squared_error(test_y_real, predictions_real))

print(f"\nSARIMAX V2 - MAE Final: {mae:.2f}€")
print(f"SARIMAX V2 - RMSE Final: {rmse:.2f}€")

### 4. Visualización

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(ts_df.index[:split_idx], ts_df['Expenses'].iloc[:split_idx], label='Entrenamiento (Gastos Reales)')
plt.plot(test_y_real.index, test_y_real, label='Prueba (Gastos Reales)')
plt.plot(test_y_real.index, predictions_real, label='Predicción V2 Walk-Forward', color='red', linestyle='--')
plt.title('Evaluación Realista: SARIMAX V2 (Walk-Forward & Log1P)')
plt.legend()
plt.show()